In [3]:
import torch
import torch.nn as nn

class ExampleMLP(nn.Module):
    def __init__(self):
        super(ExampleMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)  
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  
    def forward(self, x):
        x = x.view(-1, 784)  # flatten the input data
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = ExampleMLP()

In [2]:
weights = net.state_dict()
for layer_name, layer_weights in weights.items():
    print(f"Layer: {layer_name} weights shape {layer_weights.shape} layer dtype {layer_weights.dtype}")

Layer: fc1.weight weights shape torch.Size([128, 784]) layer dtype torch.float32
Layer: fc1.bias weights shape torch.Size([128]) layer dtype torch.float32
Layer: fc2.weight weights shape torch.Size([10, 128]) layer dtype torch.float32
Layer: fc2.bias weights shape torch.Size([10]) layer dtype torch.float32


In [5]:
import pandas as pd


def get_model_size(weights):
    names   = []
    shapes  = []
    dtype   = []
    numels  = []
    bits_per_numel   = []
    bytes_per_numel = []
    mbytes_per_numel = []
    param_sizes = []
    
    total_size = 0 
    for key, values in weights.items():
        
        names.append(key)
        shapes.append(values.shape)
        dtype.append(values.dtype)
        
        value_numels = values.numel()
        numels.append(value_numels)
        
        dtype_bits = torch.finfo(values.dtype).bits
        bits_per_numel.append(dtype_bits)
        bytes_per_numel.append(dtype_bits/8)
        
        param_sizes.append(dtype_bits/8 * value_numels)
        
        mbytes =  (dtype_bits * value_numels) / 1024**2
        mbytes_per_numel.append(mbytes )
        total_size+=mbytes
        
        
    print('model size: {:.3f}MB'.format(total_size))


    weights_df = pd.DataFrame.from_dict({"layer_name": names, "layer_shape": shapes,\
                                         "params": numels,"dtype": dtype, \
                                         "bits_per_param": bits_per_numel, 
                                         "bytes_per_param": bytes_per_numel, \
                                         "params_in_bytes": param_sizes,"params_in_mbytes": mbytes_per_numel})
    return weights_df

In [6]:
weights = net.state_dict()
net_weights = get_model_size(weights)
net_weights

model size: 3.106MB


,layer_name,layer_shape,params,dtype,bits_per_param,bytes_per_param,params_in_bytes,params_in_mbytes
0,fc1.weight,"(128, 784)",100352,torch.float32,32,4.0,401408.0,3.062500
1,fc1.bias,"(128,)",128,torch.float32,32,4.0,512.0,0.003906
2,fc2.weight,"(10, 128)",1280,torch.float32,32,4.0,5120.0,0.039062
3,fc2.bias,"(10,)",10,torch.float32,32,4.0,40.0,0.000305


In [9]:
import os

torch.save(weights, 'fp32.path')
print("FP32 size: ", os.path.getsize("fp32.path") / 1e3, "KB")

FP32 size:  409.279 KB


In [10]:
net.half()
weights = net.state_dict()
net_half_weights = get_model_size(weights)
net_half_weights

model size: 1.553MB


,layer_name,layer_shape,params,dtype,bits_per_param,bytes_per_param,params_in_bytes,params_in_mbytes
0,fc1.weight,"(128, 784)",100352,torch.float16,16,2.0,200704.0,1.531250
1,fc1.bias,"(128,)",128,torch.float16,16,2.0,256.0,0.001953
2,fc2.weight,"(10, 128)",1280,torch.float16,16,2.0,2560.0,0.019531
3,fc2.bias,"(10,)",10,torch.float16,16,2.0,20.0,0.000153


In [11]:
torch.save(weights, 'fp16.pth')
print("FP16 size: ", os.path.getsize("fp16.pth") / 1e3, "KB")

FP16 size:  205.759 KB


In [12]:
float_range_min = -1.0
float_range_max = 1.0

quant_range_min = 0
quant_range_max = 255

zero_point = 128

class Quantize:

    def __init__(self, zero_point, float_range_min, \
                 float_range_max, quant_range_min, \
                 quant_range_max):
        
        self.zero_point = zero_point
        self.scale = (float_range_max - float_range_min) / (quant_range_max - quant_range_min)

    def quantize(self, float_input: float) -> int:
        q = round((float_input / self.scale) + self.zero_point) 
        return q

    def reverse(self, int_input: int) -> float:
        q = round((int_input - self.zero_point) * self.scale,2)
        return q



quant = Quantize(zero_point, float_range_min, float_range_max, quant_range_min, quant_range_max)
print(f"Quantized value for float {0.042} is {quant.quantize(0.042)}")
print(f"Reverse Quantized value for float {133} is {quant.reverse(133)}")

Quantized value for float 0.042 is 133
Reverse Quantized value for float 133 is 0.04


In [13]:
float_range_min = -1.0
float_range_max = 1.0


zero_point = 128

class Int8Quantize:

    def __init__(self, float_range_min: float,float_range_max: float):

        quant_range_min = torch.tensor(0)
        quant_range_max = torch.tensor(255)

        float_range_min = torch.tensor(float_range_min)
        float_range_max = torch.tensor(float_range_max)

        self.scale = (float_range_max - float_range_min) / (quant_range_max - quant_range_min)
        
        self.zero_point = quant_range_min + (-float_range_min) / self.scale

    def quantize(self, float_input: float) -> int:
        q = ((float_input / self.scale) + self.zero_point).to(torch.int8)
        return q

    def reverse(self, int_input: int) -> float:
        q = ((int_input - self.zero_point) * self.scale).to(torch.float32)
        return q



quant = Int8Quantize(float_range_min, float_range_max)
print(f"Quantized value for float {0.042} is {quant.quantize(0.042)}")
print(f"Reverse Quantized value for float {133} is {quant.reverse(133)}")

Quantized value for float 0.042 is -124
Reverse Quantized value for float 133 is 0.0431373156607151


In [15]:
import torch
import torch.nn as nn

class ExampleMLP(nn.Module):
    def __init__(self):
        super(ExampleMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)  
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  
    def forward(self, x):
        x = x.view(-1, 784)  # flatten the input data
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

net = ExampleMLP()
weights = net.state_dict()

In [16]:
layer1 = weights["fc1.weight"]

min_layer1 = layer1.min()
max_layer1 = layer1.max()

print(f"Min {min_layer1} Max {max_layer1}")

quant = Int8Quantize(min_layer1, max_layer1)
quantized_weights = quant.quantize(layer1)
quantized_weights

Min -0.03571423888206482 Max 0.035713933408260345


/tmp/ipykernel_12440/2461741981.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  float_range_min = torch.tensor(float_range_min)
/tmp/ipykernel_12440/2461741981.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  float_range_max = torch.tensor(float_range_max)


tensor([[  32,  -51,   77,  ...,  -11,   93,   14],
        [  38,  -86,   53,  ...,  -60,  -62,  -81],
        [  29,   74,   76,  ..., -110,  115,   39],
        ...,
        [ -32,  107,   47,  ...,  -52,  -83,   68],
        [ -30,  -68,   63,  ...,    4,  118, -118],
        [  93,   94,   86,  ...,   48,   71,   17]], dtype=torch.int8)

In [17]:
layer1

tensor([[-0.0267,  0.0220, -0.0139,  ...,  0.0332, -0.0096, -0.0316],
        [-0.0250,  0.0121, -0.0206,  ...,  0.0194,  0.0189,  0.0134],
        [-0.0274, -0.0148, -0.0144,  ...,  0.0053, -0.0034, -0.0246],
        ...,
        [ 0.0273, -0.0055, -0.0225,  ...,  0.0217,  0.0130, -0.0165],
        [ 0.0278,  0.0170, -0.0179,  ..., -0.0344, -0.0025,  0.0032],
        [-0.0095, -0.0092, -0.0114,  ..., -0.0222, -0.0156, -0.0308]])

In [18]:
import torch
import torch.nn as nn
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

train_dataset = datasets.MNIST(root='./data', train=True, download=True)
image_sample,label = train_dataset[0]

class ExampleMLP(nn.Module):
    def __init__(self):
        super(ExampleMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)  
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  
    def forward(self, x):
        x = x.view(-1, 784)  # flatten the input data
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


to_tensor = transforms.ToTensor()

sample_tensor = to_tensor(image_sample)
sample_tensor = sample_tensor.reshape((1,-1))


net_fp32 = ExampleMLP()

output = net_fp32(sample_tensor)
output

100%|██████████| 9.91M/9.91M [00:10<00:00, 952kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 99.2kB/s]
100%|██████████| 1.65M/1.65M [00:03<00:00, 513kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.87MB/s]


tensor([[ 0.0088, -0.1482, -0.1420, -0.0884,  0.0421, -0.1234,  0.0110,  0.0984,
         -0.1173,  0.1103]], grad_fn=<AddmmBackward0>)

In [19]:
net_int8 = torch.quantization.quantize_dynamic(
    net_fp32,                  # the model to quantize
    {nn.Linear},                 # layers to quantize
    dtype=torch.qint8            # use 8-bit integers
)

quant_output = net_int8(sample_tensor)
quant_output

/tmp/ipykernel_12440/3554745793.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  net_int8 = torch.quantization.quantize_dynamic(
/opt/conda/lib/python3.12/site-packages/torch/ao/nn/quantized/modules/utils.py:72: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8,

tensor([[ 0.0085, -0.1490, -0.1420, -0.0887,  0.0418, -0.1232,  0.0116,  0.0986,
         -0.1171,  0.1096]])

In [20]:
import os

torch.save(net_fp32.state_dict(), "fp32.pth")
torch.save(net_int8.state_dict(), "int8.pth")

print("FP32 size: ", os.path.getsize("fp32.pth") / 1e3, "KB")
print("INT8 size: ", os.path.getsize("int8.pth") / 1e3, "KB")

FP32 size:  409.279 KB
INT8 size:  105.763 KB


In [22]:
class Int8Quantize:

    def __init__(self, float_range_min: float,float_range_max: float):

        quant_range_min = torch.tensor(0)
        quant_range_max = torch.tensor(255)

        float_range_min = torch.tensor(float_range_min)
        float_range_max = torch.tensor(float_range_max)

        self.scale = (float_range_max - float_range_min) / (quant_range_max - quant_range_min)
        
        self.zero_point = quant_range_min + (-float_range_min) / self.scale

    def quantize(self, float_input: float) -> int:
        q = ((float_input / self.scale) + self.zero_point).to(torch.int8)
        return q

    def reverse(self, int_input: int) -> float:
        q = ((int_input - self.zero_point) * self.scale).to(torch.float32)
        return q

In [23]:
import warnings
warnings.simplefilter("ignore")

weights_fp32 = net_fp32.fc1.weight

min_float_value = weights_fp32.min()
max_float_value = weights_fp32.max()
quant = Int8Quantize(min_float_value, max_float_value)

print(quant.quantize(weights_fp32))

tensor([[  94,   34,   29,  ..., -108,  -82,   34],
        [ -78,   55,  -62,  ...,  -24,   86,   15],
        [ -11,    1,  -59,  ...,   62,  -97,  -88],
        ...,
        [ -74,  112, -121,  ...,   84,   35,  -81],
        [  22,  103,   36,  ...,  112, -109, -124],
        [-101,   83,  105,  ...,   99,   98,  125]], dtype=torch.int8)


In [24]:
import torch
import torch.nn as nn
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

train_dataset = datasets.MNIST(root='./data', train=True, download=True)
image_sample,label = train_dataset[0]

class ExampleMLP(nn.Module):
    def __init__(self):
        super(ExampleMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)  
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  
    def forward(self, x):
        x = x.view(-1, 784)  # flatten the input data
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


to_tensor = transforms.ToTensor()

sample_tensor = to_tensor(image_sample)
sample_tensor = sample_tensor.reshape((1,-1))


net_fp32 = ExampleMLP()
output = net_fp32(sample_tensor)

weights_fp32 = net_fp32.fc1.weight

In [25]:
import torch.quantization.observer as observer

obs = observer.MinMaxObserver()

obs(weights_fp32)

# Get recorded min, max
print("Min:", obs.min_val)
print("Max:", obs.max_val)

# Calculate scale and zero point
scale, zero_point = obs.calculate_qparams()
print("Scale:", scale)
print("Zero Point:", zero_point)

Min: tensor(-0.0357)
Max: tensor(0.0357)
Scale: tensor([0.0003])
Zero Point: tensor([128], dtype=torch.int32)


In [26]:
import torch
import torch.nn as nn


class ExampleMLP(nn.Module):
    def __init__(self):
        super(ExampleMLP, self).__init__()
        self.fc1 = nn.Linear(784, 128)  
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  
    def forward(self, x):
        x = x.view(-1, 784)  # flatten the input data
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [27]:
net_fp32 = ExampleMLP()
net_fp32.eval()

net_fp32.qconfig = torch.quantization.get_default_qconfig('qnnpack')

In [28]:
net_fp32_fused = torch.quantization.fuse_modules(
    net_fp32, [['fc1', 'relu']], inplace=False
)

In [29]:
net_prepared = torch.quantization.prepare(net_fp32_fused)

In [30]:
for _ in range(100):
    example_data = torch.randn(1, 784)
    net_prepared(example_data)

In [31]:
net_int8 = torch.quantization.convert(net_prepared)

In [32]:
import torch
import torch.nn as nn
import torch.quantization

class SimpleQATModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SimpleQATModel()
model.train()

SimpleQATModel(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)

In [33]:
model_fused = torch.quantization.fuse_modules(model, [['fc1', 'relu']])
model_fused.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(model_fused, inplace=True)

SimpleQATModel(
  (fc1): LinearReLU(
    in_features=784, out_features=256, bias=True
    (weight_fake_quant): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.qint8, quant_min=-128, quant_max=127, qscheme=torch.per_channel_symmetric, reduce_range=False
      (activation_post_process): MovingAveragePerChannelMinMaxObserver(min_val=tensor([]), max_val=tensor([]))
    )
    (activation_post_process): FusedMovingAvgObsFakeQuantize(
      fake_quant_enabled=tensor([1]), observer_enabled=tensor([1]), scale=tensor([1.]), zero_point=tensor([0], dtype=torch.int32), dtype=torch.quint8, quant_min=0, quant_max=127, qscheme=torch.per_tensor_affine, reduce_range=True
      (activation_post_process): MovingAverageMinMaxObserver(min_val=inf, max_val=-inf)
    )
  )
  (relu): Identity()
  (fc2): Linear(
    in_features=256, out_features=10, bias=True
    (weight_fake_quant): Fus

In [34]:
model

SimpleQATModel(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)

In [36]:
import torch
from torch.quantization import FakeQuantize, MinMaxObserver

# Create a FakeQuantize module
fake_quant = FakeQuantize(observer=MinMaxObserver, quant_min=0, quant_max=255)

# Simulate calibration (like QAT prepare does)
fake_quant.enable_observer()
fake_quant.enable_fake_quant()

# Dummy calibration data
x = torch.tensor([[-2.0, -1.0, 0.0, 1.0, 2.0]])
fake_quant(x)  # This updates observer

# Now simulate fake quantization on a new input
x_new = torch.tensor([[0.4, 1.2, -0.9, 0.0]])
x_q = fake_quant(x_new)

print("Input:", x_new)
print("Fake quantized output:", x_q)

Input: tensor([[ 0.4000,  1.2000, -0.9000,  0.0000]])
Fake quantized output: tensor([[ 0.3922,  1.1922, -0.8941,  0.0000]])


In [46]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.Lambda(lambda x: x.view(-1)),   # flatten to 784
])

train_dataset = datasets.MNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fused.parameters(), lr=1e-3)
epochs = 3

In [47]:
model_fused.train()
for epoch in range(epochs):
    running_loss = 0.0
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model_fused(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch + 1}/{epochs}, loss: {running_loss / len(train_loader):.4f}")

Epoch 1/3, loss: 0.2586
Epoch 2/3, loss: 0.0983
Epoch 3/3, loss: 0.0656


In [48]:
model_fused.eval()
quantized_model = torch.quantization.convert(model_fused, inplace=False)